In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print(train.shape, test.shape)
train.head(10)

(2000, 8) (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
5,6,Identify the correct statement: What is the ef...,"An electric field, precisely aligned with the ...","A magnetic field, randomly aligned with the sp...","A magnetic field, precisely aligned with the s...","A gravitational field, randomly aligned with t...","A gravitational field, precisely aligned with ...",C
6,7,"Which of the following is correct? What is a ""...",A type of coffee that is made by boiling coffe...,A pattern left by a particle-laden liquid afte...,A type of coffee that is made by mixing instan...,A type of coffee that is made by pouring hot w...,A pattern left by a particle-laden liquid afte...,E
7,8,Select the most accurate option: What is the L...,The Liouville density is a probability distrib...,The Liouville density is a quasiprobability di...,The Liouville density is a bounded probability...,The Liouville density is a probability distrib...,The Liouville density is a probability distrib...,A
8,9,Select the most accurate option: What are the ...,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.,A
9,10,Which of the following is correct? What is the...,"The Wigner function W(x, p) is the Wigner tran...","The Wigner function W(x, p) is a source functi...","The Wigner function W(x, p) is the derivative ...","The Wigner function W(x, p) represents the Ham...","The Wigner function W(x, p) is the time deriva...",A


In [4]:
# feature engineering steps 
def get_overlap(prompt, option):
    prompt_words = set(prompt.lower().split())
    option_words = set(option.lower().split())
    return len(prompt_words & option_words)

def add_features(df):
    df = df.copy()

    # length features
    df['prompt_len'] = df['prompt'].str.len()
    for col in ['A', 'B', 'C', 'D', 'E']:
        df[col + '_len'] = df[col].str.len()
    df['avg_option_len'] = df[['A_len','B_len','C_len','D_len','E_len']].mean(axis=1)
    df['max_option_len'] = df[['A_len','B_len','C_len','D_len','E_len']].max(axis=1)

    # word count features
    df['prompt_word_count'] = df['prompt'].str.split().apply(len)
    for col in ['A', 'B', 'C', 'D', 'E']:
        df[col + '_word_count'] = df[col].str.split().apply(len)

    # word overlap features
    for col in ['A', 'B', 'C', 'D', 'E']:
        df[col + '_overlap'] = df.apply(lambda row: get_overlap(row['prompt'], row[col]), axis=1)

    # tfidf cosine similarity between prompt and each option
    sim_scores = {c: [] for c in ['A','B','C','D','E']}
    tfidf = TfidfVectorizer(stop_words='english')
    for i, row in df.iterrows():
        texts = [row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]
        tfidf_matrix = tfidf.fit_transform(texts)
        prompt_vec = tfidf_matrix[0]
        for j, col in enumerate(['A','B','C','D','E']):
            option_vec = tfidf_matrix[j+1]
            sim = cosine_similarity(prompt_vec, option_vec)[0][0]
            sim_scores[col].append(sim)
    for col in ['A','B','C','D','E']:
        df[col + '_sim'] = sim_scores[col]

    return df

In [5]:
train_fe = add_features(train)
train_fe.head(10)

,id,prompt,A,B,C,D,E,answer,prompt_len,A_len,...,A_overlap,B_overlap,C_overlap,D_overlap,E_overlap,A_sim,B_sim,C_sim,D_sim,E_sim
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,142,307,...,6,5,8,8,5,0.103041,0.122255,0.266821,0.250725,0.055707
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,43,406,...,3,3,3,3,3,0.463851,0.424964,0.476182,0.408551,0.334486
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,182,12,...,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,129,307,...,6,5,8,8,5,0.117195,0.139048,0.303472,0.285165,0.063358
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,110,230,...,4,4,4,3,3,0.018480,0.019138,0.018915,0.105212,0.105212
5,6,Identify the correct statement: What is the ef...,"An electric field, precisely aligned with the ...","A magnetic field, randomly aligned with the sp...","A magnetic field, precisely aligned with the s...","A gravitational field, randomly aligned with t...","A gravitational field, precisely aligned with ...",C,116,56,...,1,2,2,2,2,0.000000,0.000000,0.000000,0.000000,0.000000
6,7,"Which of the following is correct? What is a ""...",A type of coffee that is made by boiling coffe...,A pattern left by a particle-laden liquid afte...,A type of coffee that is made by mixing instan...,A type of coffee that is made by pouring hot w...,A pattern left by a particle-laden liquid afte...,E,96,65,...,4,4,3,4,3,0.089455,0.087952,0.078685,0.073305,0.087952
7,8,Select the most accurate option: What is the L...,The Liouville density is a probability distrib...,The Liouville density is a quasiprobability di...,The Liouville density is a bounded probability...,The Liouville density is a probability distrib...,The Liouville density is a probability distrib...,A,89,172,...,3,3,3,3,3,0.053167,0.065726,0.067082,0.064508,0.066800
8,9,Select the most accurate option: What are the ...,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.,A,79,112,...,1,1,1,2,1,0.000000,0.000000,0.000000,0.000000,0.000000
9,10,Which of the following is correct? What is the...,"The Wigner function W(x, p) is the Wigner tran...","The Wigner function W(x, p) is a source functi...","The Wigner function W(x, p) is the derivative ...","The Wigner function W(x, p) represents the Ham...","The Wigner function W(x, p) is the time deriva...",A,148,240,...,8,8,7,6,7,0.233607,0.161929,0.185637,0.098574,0.163858


In [6]:
test_fe = add_features(test)
test_fe.head(10)

,id,prompt,A,B,C,D,E,prompt_len,A_len,B_len,...,A_overlap,B_overlap,C_overlap,D_overlap,E_overlap,A_sim,B_sim,C_sim,D_sim,E_sim
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",144,117,117,...,1,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",123,75,80,...,1,1,1,1,0,0.000000,0.000000,0.000000,0.000000,0.000000
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,142,72,134,...,3,2,3,6,2,0.176278,0.022453,0.119872,0.175395,0.017321
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,116,234,227,...,6,4,6,4,6,0.362240,0.179854,0.333731,0.179854,0.333731
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,65,244,234,...,6,6,6,6,6,0.381676,0.345915,0.387405,0.368489,0.363548
5,6,Choose the correct answer: What is the Evans b...,The Evans balance is a mechanism used to measu...,The Evans balance is a framework used to measu...,The Evans balance is a mechanism used to measu...,The Evans balance is a structure used to measu...,The Evans balance is a structure used to measu...,81,162,164,...,3,3,3,3,3,0.064699,0.055925,0.052239,0.053836,0.072180
6,7,Which of the following is correct? What is Lor...,Lorentz symmetry or Lorentz invariance is a pr...,Lorentz symmetry or Lorentz invariance is a me...,Lorentz symmetry or Lorentz invariance is a ph...,Lorentz symmetry or Lorentz invariance is a me...,Lorentz symmetry or Lorentz invariance is an e...,106,247,283,...,7,8,7,8,7,0.152321,0.138658,0.201206,0.196198,0.245410
7,8,What are the four qualitative levels of crysta...,"Holocrystalline, hypocrystalline, hypercrystal...","Holocrystalline, hypocrystalline, hypohyaline,...","Holocrystalline, hypohyaline, hypercrystalline...","Holocrystalline, hypocrystalline, hypercrystal...","Holocrystalline, hypocrystalline, hypohyaline,...",78,67,62,...,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000
8,9,What is the Maxwell's Demon thought experiment?,A thought experiment in which a demon guards a...,A thought experiment in which a demon guards a...,A thought experiment in which a demon guards a...,A thought experiment in which a demon guards a...,A thought experiment in which a demon guards a...,47,376,387,...,3,3,3,3,3,0.202234,0.189337,0.197341,0.195467,0.192408
9,10,Which of the following is correct? What is the...,"The SI base unit of time is the week, which is...","The SI base unit of time is the second, which ...","The SI base unit of time is the hour, which is...","The SI base unit of time is the day, which is ...","The SI base unit of time is the minute, which ...",101,121,123,...,8,8,8,8,8,0.261566,0.261566,0.261566,0.261566,0.261566


In [7]:
# check both have same feature columns 
train_cols = set(train_fe.columns) - {'answer'}
test_cols = set(test_fe.columns)
print("Columns match:", train_cols == test_cols)

Columns match: True


In [8]:
train_fe.to_csv('train_features.csv', index=False)
test_fe.to_csv('test_features.csv', index=False)
print("Feature Engineering Complete")

Feature Engineering Complete
